# Health Eat — 학습 → 예측 → Kaggle 제출

**실행 순서**: 위에서 아래로 셀을 순서대로 실행하세요.  
**런타임**: GPU (T4 이상) 권장 — 런타임 → 런타임 유형 변경 → T4 GPU

| 단계 | 내용 |
|---|---|
| 1 | 환경 설정 (repo clone + 패키지 설치) |
| 2 | Kaggle 인증 (Colab Secrets) |
| 3 | 데이터 다운로드 |
| 4 | 학습 (`train.py`) |
| 5 | 예측 + 제출 파일 생성 (`predict.py`) |
| 6 | Kaggle 제출 |

## 1. 환경 설정

In [ ]:
import os

REPO_URL   = "https://github.com/beomjinkim2000/Code_IT_Team_1_FirstProject.git"
PROJECT_DIR = "/content/Code_IT_Team_1_FirstProject"
BRANCH     = "main"  # 실험 브랜치 변경 시 여기만 수정

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} checkout {BRANCH}
    !git -C {PROJECT_DIR} pull origin {BRANCH}

%cd {PROJECT_DIR}

In [ ]:
!pip install -q \
    ultralytics \
    torchmetrics[detection] \
    albumentations \
    kaggle \
    pyyaml \
    tqdm \
    iterative-stratification \
    ensemble-boxes

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Kaggle 인증

**최초 1회 설정** (이후 재실행 시 자동 적용)

1. 왼쪽 사이드바 **🔑 아이콘** 클릭
2. **`KAGGLE_USERNAME`** → Kaggle 닉네임 입력 후 저장
3. **`KAGGLE_KEY`** → [Kaggle 계정](https://www.kaggle.com/settings/account) → API → **Create New Token** → kaggle.json 열어서 `key` 값 복사 후 저장
4. 각 Secret의 **노트북 액세스 허용** 토글 ON

In [ ]:
import json
import os
from google.colab import userdata

kaggle_username = userdata.get("KAGGLE_USERNAME")
kaggle_key      = userdata.get("KAGGLE_KEY")

os.environ["KAGGLE_USERNAME"] = kaggle_username
os.environ["KAGGLE_KEY"]      = kaggle_key

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
with open(f"{kaggle_dir}/kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(f"{kaggle_dir}/kaggle.json", 0o600)

print(f"Kaggle 인증 완료 — 사용자: {kaggle_username}")

## 3. 데이터 다운로드

In [ ]:
from pathlib import Path
import zipfile

COMPETITION_ID = "ai11-level1-project"

raw_data_path = Path("data/raw/ai11-level1-project")
dataset_path  = raw_data_path / "sprint_ai_project1_data"
required_dirs = [
    dataset_path / "train_images",
    dataset_path / "test_images",
    dataset_path / "train_annotations",
]

if all(d.exists() for d in required_dirs):
    print("데이터 이미 존재 — 다운로드 건너뜀")
else:
    raw_data_path.mkdir(parents=True, exist_ok=True)
    !kaggle competitions download -c {COMPETITION_ID} -p {raw_data_path}
    for zf in raw_data_path.glob("*.zip"):
        with zipfile.ZipFile(zf) as z:
            z.extractall(raw_data_path)
        zf.unlink()
    print("압축 해제 완료")

print("\n데이터 구조:")
for d in required_dirs:
    count = len(list(d.glob("*"))) if d.exists() else 0
    print(f"  {d.name}: {count}개")

## 4. 학습

`configs/default.yaml`에서 설정 조정:
- `train.epochs`: 에폭 수 (기본 50)
- `train.batch_size`: 배치 크기 (기본 16, GPU 메모리 부족 시 8로 낮추기)
- `train.lr`: 학습률 (기본 0.001)

In [ ]:
!cat configs/default.yaml

## 4-1. Drive 모델 복원

Drive에 `best_model.pt`가 있으면 복원 후 학습을 건너뜁니다.  
재학습하려면 셀 안의 `SKIP_TRAIN = True` → `False`로 바꾸세요.

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive')

DRIVE_MODEL = Path('/content/drive/MyDrive/HealthEat/checkpoints/best_model.pt')
LOCAL_MODEL = Path('outputs/checkpoints/best_model.pt')
LOCAL_MODEL.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_MODEL.exists():
    shutil.copy(DRIVE_MODEL, LOCAL_MODEL)
    print('Drive에서 best_model.pt 복원 완료')
    SKIP_TRAIN = True  # 재학습 원하면 False로 변경
else:
    print('Drive에 모델 없음 — 학습을 실행합니다')
    SKIP_TRAIN = False

In [ ]:
# ── 제출 설정 (매번 여기만 바꾸세요) ─────────────────────────────
SUBMIT_VERSION  = "v2"                          # 파일명에 붙는 버전 태그
SUBMIT_MESSAGE  = "v2 - EMA vs raw comparison"  # Kaggle 제출 메시지
SAVE_ALL_EPOCHS = False  # True: 매 에폭 epoch_N.pt 저장 (저장공간 많이 필요) / False: best_model.pt만 저장
# ─────────────────────────────────────────────────────────────────

In [ ]:
if not SKIP_TRAIN:
    save_flag = "--save-all-epochs" if SAVE_ALL_EPOCHS else ""
    !python train.py --config configs/default.yaml --run-name {SUBMIT_VERSION} {save_flag}

In [ ]:
if not SKIP_TRAIN:
    DRIVE_MODEL.parent.mkdir(parents=True, exist_ok=True)
    if LOCAL_MODEL.exists():
        shutil.copy(LOCAL_MODEL, DRIVE_MODEL)
        print(f'Drive 저장 완료: {DRIVE_MODEL}')
    else:
        print('best_model.pt 없음 — 학습이 완료됐는지 확인하세요')

    LOCAL_LOG = Path(f'outputs/logs/metrics_{SUBMIT_VERSION}.csv')
    DRIVE_LOG = DRIVE_MODEL.parent.parent / 'logs' / f'metrics_{SUBMIT_VERSION}.csv'
    if LOCAL_LOG.exists():
        DRIVE_LOG.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(LOCAL_LOG, DRIVE_LOG)
        print(f'Drive 저장 완료: {DRIVE_LOG}')
    else:
        print(f'metrics_{SUBMIT_VERSION}.csv 없음 — train.py가 정상 종료됐는지 확인하세요')


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

log_path = Path(f'outputs/logs/metrics_{SUBMIT_VERSION}.csv')
if not log_path.exists():
    print(f'metrics_{SUBMIT_VERSION}.csv 없음 — 학습을 먼저 실행하세요')
else:
    df = pd.read_csv(log_path)
    has_val_loss = 'val_box_loss' in df.columns

    nrows = 3 if has_val_loss else 2
    fig, axes = plt.subplots(nrows, 3, figsize=(18, 5 * nrows))

    axes[0,0].plot(df['epoch'], df['box_loss'], label='box')
    axes[0,0].plot(df['epoch'], df['cls_loss'], label='cls')
    axes[0,0].plot(df['epoch'], df['dfl_loss'], label='dfl')
    axes[0,0].set_title('Train Loss'); axes[0,0].set_xlabel('Epoch')
    axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

    axes[0,1].plot(df['epoch'], df['val_mAP_ema'], label='EMA', linewidth=2)
    axes[0,1].plot(df['epoch'], df['val_mAP_raw'], label='raw', linestyle='--', alpha=0.7)
    axes[0,1].set_title('mAP@0.5:0.95 (EMA vs raw)'); axes[0,1].set_xlabel('Epoch')
    axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

    axes[0,2].plot(df['epoch'], df['val_mAP_50_ema'], label='EMA', linewidth=2)
    axes[0,2].plot(df['epoch'], df['val_mAP_50_raw'], label='raw', linestyle='--', alpha=0.7)
    axes[0,2].set_title('mAP@0.5 (EMA vs raw)'); axes[0,2].set_xlabel('Epoch')
    axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3)

    axes[1,0].plot(df['epoch'], df['lr'], color='green')
    axes[1,0].set_title('Learning Rate'); axes[1,0].set_xlabel('Epoch')
    axes[1,0].set_yscale('log'); axes[1,0].grid(True, alpha=0.3)

    gain = df['val_mAP_ema'] - df['val_mAP_raw']
    axes[1,1].plot(df['epoch'], gain, color='purple')
    axes[1,1].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[1,1].set_title('EMA 이득 (mAP@0.5:0.95)'); axes[1,1].set_xlabel('Epoch')
    axes[1,1].grid(True, alpha=0.3)

    gain50 = df['val_mAP_50_ema'] - df['val_mAP_50_raw']
    axes[1,2].plot(df['epoch'], gain50, color='orange')
    axes[1,2].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[1,2].set_title('EMA 이득 (mAP@0.5)'); axes[1,2].set_xlabel('Epoch')
    axes[1,2].grid(True, alpha=0.3)

    if has_val_loss:
        for col_idx, (key, label) in enumerate([('box', 'Box'), ('cls', 'Cls'), ('dfl', 'DFL')]):
            ax = axes[2, col_idx]
            ax.plot(df['epoch'], df[f'{key}_loss'], label='train', linewidth=2)
            ax.plot(df['epoch'], df[f'val_{key}_loss'], label='val', linestyle='--', linewidth=2)
            ax.set_title(f'{label} Loss: Train vs Val'); ax.set_xlabel('Epoch')
            ax.legend(); ax.grid(True, alpha=0.3)

    plt.suptitle(f'Training Metrics — {SUBMIT_VERSION}', fontsize=13, y=1.01)
    plt.tight_layout(); plt.show()

    best_ema = df.loc[df['val_mAP_ema'].idxmax()]
    best_raw = df.loc[df['val_mAP_raw'].idxmax()]
    gain_val = best_ema['val_mAP_ema'] - best_raw['val_mAP_raw']
    gain_pct = gain_val / best_raw['val_mAP_raw'] * 100 if best_raw['val_mAP_raw'] > 0 else 0
    print(f"best(EMA) epoch={int(best_ema['epoch'])}  mAP@50:95={best_ema['val_mAP_ema']:.4f}  mAP@50={best_ema['val_mAP_50_ema']:.4f}")
    print(f"best(raw) epoch={int(best_raw['epoch'])}  mAP@50:95={best_raw['val_mAP_raw']:.4f}  mAP@50={best_raw['val_mAP_50_raw']:.4f}")
    print(f"EMA 이득: {gain_val:+.4f}  ({gain_pct:+.1f}%)")


## 4-3. 클래스별 F1 추적

best epoch 기준 클래스별 F1을 확인합니다.

| 판단 기준 | 의미 |
|---|---|
| F1 < 0.7 (빨간색) | 해당 클래스가 bottleneck |
| Precision < Recall | 다른 클래스로 오탐(FP)이 많음 |
| Recall < Precision | 알약을 못 잡음(FN) — 데이터 부족 or 해상도 문제 |

→ Q4(YOLO11x), Q6(해상도 1024), Q7(AI Hub 데이터) 실험에서 어떤 클래스가 개선됐는지 비교하세요.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

f1_path      = Path(f'outputs/logs/f1_{SUBMIT_VERSION}.csv')
metrics_path = Path(f'outputs/logs/metrics_{SUBMIT_VERSION}.csv')

if not f1_path.exists():
    print(f'f1_{SUBMIT_VERSION}.csv 없음 — 학습을 먼저 실행하세요')
else:
    df_f1 = pd.read_csv(f1_path)

    # best EMA epoch 기준
    if metrics_path.exists():
        df_m = pd.read_csv(metrics_path)
        best_epoch = int(df_m.loc[df_m['val_mAP_ema'].idxmax(), 'epoch'])
    else:
        best_epoch = int(df_f1['epoch'].max())

    df_best = df_f1[df_f1['epoch'] == best_epoch].copy()

    if df_best.empty:
        print(f'epoch={best_epoch} 데이터 없음')
    else:
        LOW_F1 = 0.7
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))

        # 클래스별 F1 바 차트 — F1 < 0.7 은 빨간색
        colors = ['red' if f < LOW_F1 else 'steelblue' for f in df_best['f1']]
        axes[0].barh(df_best['class_id'].astype(str), df_best['f1'], color=colors)
        axes[0].axvline(LOW_F1, color='red', linestyle='--', label=f'F1={LOW_F1} 기준선')
        axes[0].set_title(f'클래스별 F1 (epoch={best_epoch}, best EMA)')
        axes[0].set_xlabel('F1'); axes[0].set_ylabel('class_id')
        axes[0].legend(); axes[0].grid(True, alpha=0.3, axis='x')

        # F1 낮은 클래스의 에폭 추이 (최대 10개)
        low_classes = df_best[df_best['f1'] < LOW_F1]['class_id'].tolist()
        if low_classes:
            for cls_id in low_classes[:10]:
                sub = df_f1[df_f1['class_id'] == cls_id]
                axes[1].plot(sub['epoch'], sub['f1'], label=f'cls {cls_id}')
            axes[1].axhline(LOW_F1, color='red', linestyle='--', alpha=0.5)
            axes[1].set_title(f'F1 < {LOW_F1} 클래스 에폭 추이')
            axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1')
            axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
        else:
            axes[1].text(0.5, 0.5, f'모든 클래스 F1 ≥ {LOW_F1}', ha='center', va='center',
                         transform=axes[1].transAxes, fontsize=14, color='green')
            axes[1].set_title('F1 추이')

        plt.suptitle(f'클래스별 F1 — {SUBMIT_VERSION}', fontsize=13)
        plt.tight_layout(); plt.show()

        print(f"=== best epoch={best_epoch} 클래스별 F1 ===")
        print(f"{'cls':>5}  {'precision':>9}  {'recall':>7}  {'f1':>6}  {'판정'}")
        print("-" * 46)
        for _, row in df_best.sort_values('f1').iterrows():
            flag = "⚠️  낮음" if row['f1'] < LOW_F1 else ""
            print(f"{int(row['class_id']):>5}  {row['precision']:>9.4f}  {row['recall']:>7.4f}  {row['f1']:>6.4f}  {flag}")

        n_low = (df_best['f1'] < LOW_F1).sum()
        print(f"\nF1 < {LOW_F1} 클래스: {n_low} / {len(df_best)}개")
        if n_low > 0:
            worst = df_best.loc[df_best['f1'].idxmin()]
            print(f"최저 F1: cls {int(worst['class_id'])}  "
                  f"F1={worst['f1']:.4f}  P={worst['precision']:.4f}  R={worst['recall']:.4f}")
            if worst['precision'] < worst['recall']:
                print("  → Precision < Recall: 다른 클래스로 오탐(FP)이 많음")
            else:
                print("  → Recall < Precision: 알약을 못 잡음(FN) — 데이터 부족 or 해상도 문제")

In [ ]:
import shutil
from pathlib import Path

ckpt_dir   = Path('outputs/checkpoints')
sub_dir    = Path('outputs/submissions')
drive_sub  = Path('/content/drive/MyDrive/HealthEat/submissions')
sub_dir.mkdir(parents=True, exist_ok=True)
drive_sub.mkdir(parents=True, exist_ok=True)

ckpt = ckpt_dir / 'best_model.pt'

if not ckpt.exists():
    print('⚠️  best_model.pt 없음')
else:
    # EMA 예측
    !python predict.py --config configs/default.yaml --checkpoint {ckpt} --use-ema
    src = sub_dir / 'submission_v1.csv'
    if src.exists():
        dst = sub_dir / f'submission_{SUBMIT_VERSION}_ema.csv'
        shutil.copy(src, dst)
        shutil.copy(dst, drive_sub / dst.name)
        print(f'✅ EMA 예측 완료 → {dst.name} (Drive 저장 완료)')

    # raw 예측
    !python predict.py --config configs/default.yaml --checkpoint {ckpt}
    src = sub_dir / 'submission_v1.csv'
    if src.exists():
        dst = sub_dir / f'submission_{SUBMIT_VERSION}_raw.csv'
        shutil.copy(src, dst)
        shutil.copy(dst, drive_sub / dst.name)
        print(f'✅ raw 예측 완료 → {dst.name} (Drive 저장 완료)')

In [ ]:
import pandas as pd
from pathlib import Path

sub_dir = Path('outputs/submissions')
for label, fname in [('EMA', f'submission_{SUBMIT_VERSION}_ema.csv'),
                     ('raw', f'submission_{SUBMIT_VERSION}_raw.csv')]:
    p = sub_dir / fname
    if p.exists():
        df = pd.read_csv(p)
        det = df.groupby('image_id').size()
        print(f"[{label:>3}]  행 수: {len(df):5d}  이미지: {df['image_id'].nunique():4d}"
              f"  탐지/이미지: {det.mean():.2f}  avg score: {df['score'].mean():.4f}"
              f"  max: {df['score'].max():.4f}  min: {df['score'].min():.4f}")
    else:
        print(f'[{label:>3}]  파일 없음: {p}')


## 5-1. 예측 진단

score 분포와 이미지당 탐지 수를 확인합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# EMA 파일 우선, 없으면 raw로 fallback
csv_path = Path(f'outputs/submissions/submission_{SUBMIT_VERSION}_ema.csv')
if not csv_path.exists():
    csv_path = Path(f'outputs/submissions/submission_{SUBMIT_VERSION}_raw.csv')
if not csv_path.exists():
    print(f'submission_{SUBMIT_VERSION}_ema.csv / _raw.csv 없음 — 예측을 먼저 실행하세요')
else:
    df = pd.read_csv(csv_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df['score'], bins=50, edgecolor='black', color='steelblue')
    axes[0].axvline(0.25, color='red', linestyle='--', label='conf_threshold=0.25')
    axes[0].set_title('Score Distribution'); axes[0].set_xlabel('score'); axes[0].set_ylabel('Count')
    axes[0].legend()

    det_per_img = df.groupby('image_id').size()
    axes[1].hist(det_per_img, bins=range(0, det_per_img.max() + 2), edgecolor='black', color='steelblue')
    axes[1].set_title('Detections per Image'); axes[1].set_xlabel('Detections'); axes[1].set_ylabel('Images')

    plt.tight_layout(); plt.show()
    print(f"파일: {csv_path.name}")
    print(f"score  평균: {df['score'].mean():.3f}  최솟값: {df['score'].min():.3f}  최댓값: {df['score'].max():.3f}")
    print(f"이미지당 평균 탐지: {det_per_img.mean():.1f}  최대: {det_per_img.max()}  검출된 이미지 수: {len(det_per_img)}")

In [ ]:
import sys; sys.path.insert(0, '.')
import torch, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from src.utils.config import load_config
from src.data.dataset import PillDataset, RAW_DATA_ROOT
from src.data.split import train_val_split
from src.data.transforms import val_transform
from src.engine.predict import predict_batch
from src.engine.postprocess import PostprocessConfig, postprocess_raw_outputs
from src.models.baseline import build_model

cfg = load_config()
img_size = cfg['train']['img_size']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt = torch.load('outputs/checkpoints/best_model.pt', map_location=device, weights_only=True)
model = build_model(cfg['data']['nc'])
model.load_state_dict(ckpt['model_state'])
model.to(device).eval()

all_files = sorted((RAW_DATA_ROOT / 'train_images').glob('*.png'))
_, val_files = train_val_split(all_files, val_ratio=cfg['split']['val_ratio'], seed=cfg['train']['seed'])
sample_files = val_files[:4]

postprocess_cfg = PostprocessConfig(**cfg['postprocess'])
transform = val_transform(img_size)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, fname in zip(axes, sample_files):
    img = Image.open(RAW_DATA_ROOT / 'train_images' / fname).convert('RGB')
    orig_w, orig_h = img.size
    t_img, _ = transform(img, {'boxes': [], 'labels': [], 'image_id': 0})
    tensor = torch.from_numpy(np.array(t_img)).permute(2,0,1).float().div(255).unsqueeze(0).to(device)
    raw = predict_batch(model, tensor, device)
    preds = postprocess_raw_outputs(raw, image_ids=[0], config=postprocess_cfg)
    ax.imshow(img)
    for box, score, label in zip(preds[0]['boxes'].cpu(), preds[0]['scores'].cpu(), preds[0]['labels'].cpu()):
        x1, y1, x2, y2 = [v * (orig_w if i % 2 == 0 else orig_h) / img_size for i, v in enumerate(box.tolist())]
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none'))
        ax.text(x1, y1-5, f'cls{label.item()} {score:.2f}', color='red', fontsize=7)
    ax.set_title(f'{fname} ({len(preds[0]["boxes"])} boxes)'); ax.axis('off')

plt.suptitle('Val Predictions (red = predicted box)', fontsize=12)
plt.tight_layout(); plt.show()

## 5-2. F1 곡선 — 최적 confidence threshold 찾기

현재 `postprocess.conf_threshold=0.25`는 관행적 기본값입니다.
이 셀은 val set 전체를 conf=0.001로 예측한 뒤 conf를 0.05→0.90으로 sweep해서
**F1이 최대인 conf**를 찾습니다.

- `best_conf`가 현재값과 ≥ 0.05 차이나면 → `configs/default.yaml`에 반영 후 재제출
- `skip_box_thr` (WBF)도 같은 방식으로 최적화 가능 ([[concepts/Confidence Score]])

In [ ]:
# 이 셀은 위 '5-1. 예측 진단' 셀 실행 후 실행하세요 (model, val_files, cfg 공유)
from src.engine.evaluate import compute_per_class_f1
from src.engine.postprocess import PostprocessConfig
import numpy as np

nc           = cfg['data']['nc']
current_conf = cfg['postprocess']['conf_threshold']

# val 전체 예측 수집 — conf=0.001 로 낮춰야 PR 곡선 전체 범위 커버
raw_post_cfg = PostprocessConfig(
    conf_threshold=0.001,
    iou_threshold=postprocess_cfg.iou_threshold,
    max_detections=postprocess_cfg.max_detections,
)

from src.data.dataset import PillDataset
dataset = PillDataset(
    image_files=val_files,
    data_root=RAW_DATA_ROOT / 'train_images',
    transform=transform,
    annotation_dir=RAW_DATA_ROOT / 'train_annotations',
)
loader = torch.utils.data.DataLoader(
    dataset, batch_size=8, shuffle=False,
    collate_fn=lambda batch: list(zip(*batch))
)

print("val set 예측 수집 중 (conf=0.001)...")
all_preds, all_targets = [], []
model.eval()
with torch.no_grad():
    for images, targets in loader:
        imgs = torch.stack([
            torch.from_numpy(np.array(img)).permute(2, 0, 1).float().div(255)
            for img in images
        ]).to(device)
        raw  = predict_batch(model, imgs, device)
        preds = postprocess_raw_outputs(raw, image_ids=[t['image_id'] for t in targets], config=raw_post_cfg)
        all_preds.extend(preds)
        all_targets.extend(targets)
print(f"수집 완료: {len(all_preds)}개 이미지")

# conf sweep 0.05 → 0.90
conf_range = np.arange(0.05, 0.91, 0.05)
f1_vals, prec_vals, rec_vals = [], [], []
print("F1 곡선 계산 중...")
for conf in conf_range:
    stats = compute_per_class_f1(all_preds, all_targets, num_classes=nc, conf_threshold=float(conf))
    if stats:
        f1_vals.append(np.mean([v['f1']        for v in stats.values()]))
        prec_vals.append(np.mean([v['precision'] for v in stats.values()]))
        rec_vals.append(np.mean([v['recall']    for v in stats.values()]))
    else:
        f1_vals.append(0.0); prec_vals.append(0.0); rec_vals.append(0.0)

best_idx  = int(np.argmax(f1_vals))
best_conf = float(conf_range[best_idx])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(conf_range, f1_vals,   label='F1',        linewidth=2,   color='blue')
axes[0].plot(conf_range, prec_vals, label='Precision',  linestyle='--', color='green')
axes[0].plot(conf_range, rec_vals,  label='Recall',     linestyle='--', color='orange')
axes[0].axvline(best_conf,    color='blue', linestyle=':',  linewidth=1.5, label=f'최적 conf={best_conf:.2f}')
axes[0].axvline(current_conf, color='red',  linestyle='--', linewidth=1.5, label=f'현재 conf={current_conf:.2f}')
axes[0].set_title('F1 / Precision / Recall vs conf threshold')
axes[0].set_xlabel('conf threshold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(rec_vals, prec_vals, 'b-o', markersize=4)
axes[1].set_title('PR 곡선 (conf sweep)')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'F1 곡선 — 최적 conf threshold 탐색 ({SUBMIT_VERSION})', fontsize=13)
plt.tight_layout(); plt.show()

print(f"\n현재  conf_threshold : {current_conf:.2f}  (configs/default.yaml postprocess.conf_threshold)")
print(f"최적  conf_threshold : {best_conf:.2f}  (F1={f1_vals[best_idx]:.4f})")
if abs(best_conf - current_conf) >= 0.05:
    print(f"⚠️  default.yaml의 postprocess.conf_threshold를 {best_conf:.2f}로 바꾸고 재제출 권장")
else:
    print(f"✅  현재 conf={current_conf:.2f}이 최적과 가깝습니다 — 변경 불필요")

## 6. Kaggle 제출

In [ ]:
# ── 제출 모델 선택 ─────────────────────────────────────────────────
# 위 비교 확인 후 선택하세요
USE_EMA = True   # True: EMA 모델  /  False: 원본 모델
# ─────────────────────────────────────────────────────────────────

from pathlib import Path
model_label = "EMA" if USE_EMA else "raw"
chosen_csv  = Path(f'outputs/submissions/submission_{SUBMIT_VERSION}_{model_label}.csv')

if chosen_csv.exists():
    import pandas as pd
    df = pd.read_csv(chosen_csv)
    print(f'제출 파일: {chosen_csv.name}')
    print(f'행 수: {len(df)}  /  이미지 수: {df["image_id"].nunique()}')
    df.head(5)
else:
    print(f'⚠️  {chosen_csv} 없음 — 예측 셀을 먼저 실행하세요')

In [ ]:
!kaggle competitions submit \
    -c {COMPETITION_ID} \
    -f {chosen_csv} \
    -m "{SUBMIT_MESSAGE} [{model_label}]"

print(f'제출 완료! ({model_label} 모델)')


In [ ]:
!kaggle competitions submissions -c {COMPETITION_ID}